# Notebook A — EEG Spectral : Boxplots par bande, canal et groupe

**Input :** `c:\dev\Cerco_studies\data\eeg_features.csv` (sortie de `04_extract_features.py`)  
**Input :** `c:\dev\Cerco_studies\data\patients_label.txt`  
**Output :** figures dans `c:\dev\Cerco_studies\data\visualisation\results\eeg\boxplots` + tables stats dans `c:\dev\Cerco_studies\data\visualisation\results\eeg\stats`

Ce notebook produit :
1. Boxplots (puissance absolue + relative) par bande fréquentielle × canal bipolaire × groupe clinique
2. Tests statistiques : Shapiro → ANOVA+Tukey **ou** Kruskal+Mann-Whitney (Bonferroni/Holm)
3. Export Excel récapitulatif avec étoiles de significativité

In [ ]:
# ============================================================
# PARAMÈTRES  
# ============================================================
EEG_FEATURES_CSV = r"c:\dev\Cerco_studies\data\eeg_features.csv"
LABELS_TXT       = r"c:\dev\Cerco_studies\data\patients_label.txt"
OUTPUT_FIGS      = r"c:\dev\Cerco_studies\data\visualisation\eeg\boxplots"
OUTPUT_STATS     = r"c:\dev\Cerco_studies\data\visualisation\eeg\stats"

SAVE_FIGS  = True    # False = affiche sans sauvegarder
FIG_FORMAT = "pdf"   # "pdf", "png" ou "svg"
DPI        = 300
ALPHA      = 0.05

# Palette 4 groupes cliniques
PALETTE = {
    "EAI":   "#C9D175",
    "Narco": "#F15854",
    "SYN":   "#44AA99",
    "TCSPi": "#BEBEBE",
}
GROUP_ORDER = ["SYN", "Narco", "TCSPi", "EAI"]

BANDS = ["delta", "theta", "alpha", "beta", "gamma_bas", "gamma_haut"]
BAND_FREQS = {
    "delta":      "0.5–4 Hz",
    "theta":      "4–8 Hz",
    "alpha":      "8–12 Hz",
    "beta":       "12–30 Hz",
    "gamma_bas":  "30–50 Hz",
    "gamma_haut": "50–80 Hz",
}
BIPOLAR_CHANNELS = [
    "Fp1-T3", "Fp1-C3", "T3-O1",
    "Fp2-T4", "Fp2-C4", "T4-O2",
    "Fp1-A1", "Fp2-A1", "T3-A1",
    "C3-A1",  "T4-A1",  "C4-A1",
]

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations

import scipy.stats as st
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

Path(OUTPUT_FIGS).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_STATS).mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
matplotlib.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
})
print("Librairies chargées.")

NameError: name 'OUTPUT_FIGS' is not defined

In [ ]:
# ============================================================
# CHARGEMENT ET FUSION
# ============================================================
feat = pd.read_csv(EEG_FEATURES_CSV)
feat["patient_id"] = feat["patient_id"].astype(str).str.strip()

# Labels
try:
    lbl = pd.read_csv(LABELS_TXT)
    cols = {c.lower(): c for c in lbl.columns}
    id_col  = cols.get("patient_id") or cols.get("identifiant")
    lbl_col = cols.get("label_str") or cols.get("diagnostic") or cols.get("label")
    lbl = lbl[[id_col, lbl_col]].rename(columns={id_col: "patient_id", lbl_col: "group"})
except Exception:
    lbl = pd.read_csv(LABELS_TXT, header=None, names=["patient_id", "group"])

lbl["patient_id"] = lbl["patient_id"].astype(str).str.strip()
lbl["group"]      = lbl["group"].astype(str).str.strip()

# Mapping vers macro-classes
def to_macro(s):
    u = s.upper()
    if any(k in u for k in ("PARK","MPI","AMS","DCL","DLB","PAF")): return "SYN"
    if "NARCO" in u: return "Narco"
    if "TCSP" in u or "RBDI" in u: return "TCSPi"
    if "EAI" in u or "ENCEPHALITE" in u: return "EAI"
    return s

lbl["group"] = lbl["group"].map(to_macro)

df = feat.merge(lbl, on="patient_id", how="left")
df_before_filter = df.copy()          
df = df[df["group"].isin(GROUP_ORDER)].copy()

excluded = df_before_filter[~df_before_filter["patient_id"].isin(df["patient_id"])]
if not excluded.empty:
    print(f"Attention : {len(excluded)} patients exclus (groupe non reconnu) :")
    print(excluded[["patient_id", "group"]].to_string())

print(f"Patients : {df['patient_id'].nunique()} | Groupes : {df['group'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# RESHAPE : une ligne = patient × canal × bande
# ============================================================
# On reconstruit un tableau long depuis les colonnes eeg_{ch}_{band}_bp_{abs|rel}_mean

records = []
for _, row in df.iterrows():
    pid   = row["patient_id"]
    group = row["group"]
    for ch in BIPOLAR_CHANNELS:
        ch_col = ch.replace("-", "-")  # les noms de colonnes utilisent le tiret
        for band in BANDS:
            # cherche les colonnes abs et rel (agrégées _mean)
            col_abs = f"eeg_{ch}_{band}_bp_abs_mean"
            col_rel = f"eeg_{ch}_{band}_bp_rel_mean"
            val_abs = row.get(col_abs, np.nan)
            val_rel = row.get(col_rel, np.nan)
            records.append({
                "patient_id": pid,
                "group": group,
                "channel": ch,
                "band": band,
                "abs": val_abs,
                "rel": val_rel,
            })

long_df = pd.DataFrame(records).dropna(subset=["group"])
print(f"Tableau long : {long_df.shape} | Canaux : {long_df['channel'].nunique()} | Bandes : {long_df['band'].nunique()}")

In [ ]:
# ============================================================
# HELPERS STATISTIQUES
# ============================================================
def iqr_bounds(a):
    q1, q3 = np.percentile(a, [25, 75])
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

def p_stars(p):
    if pd.isna(p): return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return ""

def run_stats_block(data, value_col, group_col="group"):
    """Retourne (normality_ok, kw_p, pairwise_df)."""
    groups = {g: sub[value_col].dropna().values
              for g, sub in data.groupby(group_col) if len(sub) >= 3}
    if len(groups) < 2:
        return False, np.nan, pd.DataFrame()

    # Shapiro sur chaque groupe
    normal = all(
        st.shapiro(v)[1] >= ALPHA
        for v in groups.values() if 3 <= len(v) <= 5000
    )

    # Kruskal–Wallis
    kw_stat, kw_p = st.kruskal(*groups.values())

    # Post-hoc Mann–Whitney + Holm
    pw_rows = []
    for (g1, x), (g2, y) in combinations(groups.items(), 2):
        u, p_raw = st.mannwhitneyu(x, y, alternative="two-sided")
        pw_rows.append({"group1": g1, "group2": g2,
                        "n1": len(x), "n2": len(y),
                        "U": u, "p_raw": p_raw,
                        "r_rb": 1 - 2*u/(len(x)*len(y))})

    pw = pd.DataFrame(pw_rows)
    if not pw.empty:
        _, p_holm, _, _ = multipletests(pw["p_raw"], method="holm")
        pw["p_holm"] = p_holm
        pw["stars"]  = pw["p_holm"].map(p_stars)

    return normal, kw_p, pw

print("Helpers statistiques prêts.")

In [ ]:
# ============================================================
# HELPER BOXPLOT
# ============================================================
def boxplot_band_channel(data, channel, band, value_col, ylabel, title, save_path=None):
    sub = data[(data["channel"] == channel) & (data["band"] == band)].copy()
    if sub.empty:
        return

    # Filtrage IQR par groupe
    filtered = []
    for g, grp in sub.groupby("group"):
        lo, hi = iqr_bounds(grp[value_col].dropna().values)
        filtered.append(grp[(grp[value_col] >= lo) & (grp[value_col] <= hi)])
    sub_f = pd.concat(filtered, ignore_index=True)

    # Stats
    _, kw_p, pw = run_stats_block(sub_f, value_col)

    fig, ax = plt.subplots(figsize=(7, 5))

    order = [g for g in GROUP_ORDER if g in sub_f["group"].unique()]
    sns.boxplot(data=sub_f, x="group", y=value_col, order=order,
                palette=PALETTE, showfliers=False, ax=ax, legend=False)
    sns.stripplot(data=sub_f, x="group", y=value_col, order=order,
                  color="black", alpha=0.55, size=4, jitter=True, ax=ax)

    # Annotations significatives (Holm)
    if not pw.empty:
        sig = pw[pw["stars"] != ""]
        ymax = sub_f[value_col].max()
        yrange = sub_f[value_col].max() - sub_f[value_col].min()
        step = yrange * 0.12
        for k, (_, row) in enumerate(sig.iterrows()):
            x1 = order.index(row["group1"])
            x2 = order.index(row["group2"])
            y_ = ymax + step*(k+1)
            ax.plot([x1, x1, x2, x2], [y_-step*0.2, y_, y_, y_-step*0.2],
                    lw=1.2, color="black")
            ax.text((x1+x2)/2, y_, row["stars"], ha="center", va="bottom",
                    fontsize=13, fontweight="bold")

    kw_label = f"Kruskal–Wallis p={kw_p:.3f}" if not np.isnan(kw_p) else ""
    ax.set_title(f"{title}\n{channel} | {BAND_FREQS.get(band, band)}\n{kw_label}",
                 fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    plt.tight_layout()

    if save_path and SAVE_FIGS:
        fig.savefig(save_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Helper boxplot prêt.")

In [ ]:
# ============================================================
# FIGURE 1 — Puissance RELATIVE par bande × canal
# ============================================================
for band in BANDS:
    for ch in BIPOLAR_CHANNELS:
        save_path = Path(OUTPUT_FIGS) / f"bp_rel_{band}_{ch.replace('-','')}.{FIG_FORMAT}"
        boxplot_band_channel(
            long_df, channel=ch, band=band,
            value_col="rel",
            ylabel="Relative power (a.u.)",
            title="Relative spectral power",
            save_path=save_path,
        )

In [ ]:
# ============================================================
# FIGURE 2 — Puissance ABSOLUE par bande × canal
# ============================================================
for band in BANDS:
    for ch in BIPOLAR_CHANNELS:
        save_path = Path(OUTPUT_FIGS) / f"bp_abs_{band}_{ch.replace('-','')}.{FIG_FORMAT}"
        boxplot_band_channel(
            long_df, channel=ch, band=band,
            value_col="abs",
            ylabel="Absolute power (V²/Hz)",
            title="Absolute spectral power",
            save_path=save_path,
        )

In [ ]:
# ============================================================
# TESTS STATISTIQUES COMPLETS — Export CSV + Excel
# ============================================================
kw_rows, pw_rows, means_rows = [], [], []

for band in BANDS:
    for ch in BIPOLAR_CHANNELS:
        for vcol in ["rel", "abs"]:
            sub = long_df[(long_df["channel"]==ch) & (long_df["band"]==band)].copy()
            if sub["group"].nunique() < 2:
                continue

            # Moyennes descriptives
            for g, grp in sub.groupby("group"):
                vals = grp[vcol].dropna()
                means_rows.append({"band": band, "channel": ch, "value": vcol,
                                   "group": g, "mean": vals.mean(),
                                   "sd": vals.std(), "n": len(vals)})

            # Kruskal + post-hoc
            _, kw_p, pw = run_stats_block(sub, vcol)
            kw_rows.append({"band": band, "channel": ch, "value": vcol, "p_kw": kw_p})
            if not pw.empty:
                for _, r in pw.iterrows():
                    pw_rows.append({"band": band, "channel": ch, "value": vcol,
                                    **r.to_dict()})

means_df = pd.DataFrame(means_rows)
kw_df    = pd.DataFrame(kw_rows)
pw_df    = pd.DataFrame(pw_rows)

means_df.to_csv(Path(OUTPUT_STATS)/"eeg_means_per_band_channel_group.csv", index=False)
kw_df.to_csv(Path(OUTPUT_STATS)/"eeg_kruskal_results.csv", index=False)
pw_df.to_csv(Path(OUTPUT_STATS)/"eeg_mannwhitney_posthoc.csv", index=False)

# Excel récap — seulement significatif
sig_df = pw_df[pw_df["stars"] != ""].copy() if not pw_df.empty else pd.DataFrame()
if not sig_df.empty:
    sig_df.to_excel(Path(OUTPUT_STATS)/"eeg_significant_comparisons.xlsx", index=False)
    print(f"{len(sig_df)} comparaisons significatives exportées.")
else:
    print("Aucune comparaison significative.")

print("Stats exportées dans", OUTPUT_STATS)

In [ ]:
# ============================================================
# FIGURE 3 — Heatmap de significativité (p_holm) par bande × canal
# ============================================================
if not pw_df.empty:
    for vcol in ["rel", "abs"]:
        sub = pw_df[pw_df["value"]==vcol].copy()
        if sub.empty:
            continue
        sub["pair"] = sub["group1"] + " vs " + sub["group2"]
        pivot = sub.pivot_table(index="pair", columns=["channel","band"],
                                values="p_holm", aggfunc="min")

        fig, ax = plt.subplots(figsize=(max(14, len(pivot.columns)*0.6), max(4, len(pivot)*0.8)))
        sns.heatmap(-np.log10(pivot.fillna(1).clip(lower=1e-10)),
                    cmap="YlOrRd", ax=ax, linewidths=0.3,
                    cbar_kws={"label": "−log₁₀(p_holm)"})
        ax.axhline(y=-np.log10(ALPHA), color="black", linestyle="--", linewidth=1)
        ax.set_title(f"Significativité Mann–Whitney Holm — puissance {vcol}", fontsize=12)
        ax.set_xlabel("")
        ax.set_ylabel("Paires de groupes")
        plt.tight_layout()
        if SAVE_FIGS:
            fig.savefig(Path(OUTPUT_FIGS)/f"heatmap_pvalues_{vcol}.{FIG_FORMAT}",
                        dpi=DPI, bbox_inches="tight")
        plt.show()
        plt.close(fig)